<a href="https://colab.research.google.com/github/dantruongnv/AI-Email-Spam-Detector/blob/main/AI_Email_Spam_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "pandas==2.2.3" google-api-python-client google-auth-httplib2 google-auth-oauthlib scikit-learn

In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# Quyền truy cập: Đọc, chỉnh sửa nhãn thư rác/hộp thư đến
SCOPES = ['https://www.googleapis.com/auth/gmail.modify']

# 2. XÁC THỰC GMAIL API
def authenticate_gmail():
    """Xác thực người dùng OAuth2 tương thích hoàn toàn với môi trường Google Colab."""
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists('credentials.json'):
                raise FileNotFoundError(
                    "Không tìm thấy file 'credentials.json'! "
                    "Vui lòng tải file credential từ Google Cloud và kéo thả vào thư mục Files trên Colab."
                )

            # Cấu hình redirect_uri dành cho môi trường không có trình duyệt (Console/Colab)
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json',
                SCOPES,
                redirect_uri='urn:ietf:wg:oauth:2.0:oob'
            )

            auth_url, _ = flow.authorization_url(prompt='consent')

            print("\n" + "="*60)
            print("ĐÃ TẠO LINK XÁC THỰC DÀNH CHO COLAB:")
            print("1. Nhấp vào đường link dưới đây để đăng nhập Gmail:")
            print(auth_url)
            print("\n2. Cho phép ứng dụng, sau đó sao chép Mã xác thực (Authorization Code) nhận được.")
            print("="*60 + "\n")

            code = input("Dán Mã xác thực (Authorization Code) vào đây và nhấn Enter: ").strip()
            flow.fetch_token(code=code)
            creds = flow.credentials

        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    return build('gmail', 'v1', credentials=creds)


# 3. HUẤN LUYỆN MÔ HÌNH TỪ LINK DATASET TRỰC TIẾP
def train_spam_model():
    """Tải tập dữ liệu Email trực tiếp từ GitHub và huấn luyện mô hình Naive Bayes."""
    url = "https://raw.githubusercontent.com/thehananbhat/spam-vs-ham/master/spam_ham_dataset.csv"

    print("Đang tải tập dữ liệu Email từ link trực tiếp...")
    df = pd.read_csv(url)

    # Loại bỏ giá trị trống
    df = df.dropna(subset=['text', 'label'])

    X = df['text']
    y = df['label'] # Giá trị: 'spam' hoặc 'ham'

    # Chia tập dữ liệu Train / Test (80% / 20%)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print("Đang huấn luyện mô hình phân loại Spam...")
    model = make_pipeline(TfidfVectorizer(stop_words='english'), MultinomialNB())
    model.fit(X_train, y_train)

    # Đánh giá độ chính xác
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"-> Huấn luyện hoàn tất! Độ chính xác (Accuracy): {acc * 100:.2f}%\n")

    return model

# 4. LẤY EMAIL CHƯA ĐỌC TỪ GMAIL
def get_unread_emails(service, max_results=5):
    """Lấy danh sách thư chưa đọc từ hộp thư đến (INBOX)."""
    results = service.users().messages().list(
        userId='me',
        q='is:unread label:INBOX',
        maxResults=max_results
    ).execute()

    messages = results.get('messages', [])
    emails = []

    for msg in messages:
        m = service.users().messages().get(userId='me', id=msg['id'], format='full').execute()

        payload = m.get('payload', {})
        headers = payload.get('headers', [])

        subject = next((h['value'] for h in headers if h['name'] == 'Subject'), 'Không tiêu đề')
        snippet = m.get('snippet', '')

        emails.append({'id': msg['id'], 'subject': subject, 'snippet': snippet})

    return emails

# 5. PHÂN LOẠI VÀ CHUYỂN VÀO MỤC SPAM
def classify_and_move_spam(service, model):
    """Dự đoán nội dung email và di chuyển vào thư mục SPAM nếu là thư rác."""
    emails = get_unread_emails(service)
    if not emails:
        print("Không có thư mới chưa đọc trong INBOX.")
        return

    print(f"Tìm thấy {len(emails)} thư chưa đọc. Bắt đầu phân loại:")
    print("--------------------------------------------------")

    for email in emails:
        full_text = f"{email['subject']} {email['snippet']}"
        prediction = model.predict([full_text])[0]

        print(f"Subject : {email['subject']}")
        print(f"Kết quả : {prediction.upper()}")

        if prediction == 'spam':
            service.users().messages().batchModify(
                userId='me',
                body={
                    'ids': [email['id']],
                    'addLabelIds': ['SPAM'],
                    'removeLabelIds': ['INBOX']
                }
            ).execute()
            print("-> Trạng thái: Đã di chuyển vào thư mục SPAM.")
        else:
            print("-> Trạng thái: Giữ nguyên trong Hộp thư đến (INBOX).")
        print("--------------------------------------------------")

# 6. THỰC THI CHƯƠNG TRÌNH
if __name__ == '__main__':
    # Bước 1: Xác thực Gmail API
    gmail_service = authenticate_gmail()

    # Bước 2: Huấn luyện mô hình từ dataset online
    spam_classifier = train_spam_model()

    # Bước 3: Phân loại và tự động xử lý thư rác
    classify_and_move_spam(gmail_service, spam_classifier)